# Bible Corpus: Pairwise Domain Distance

Downloads selected language bibles from https://github.com/christos-c/bible-corpus and evaluates
pairwise domain-distance metrics from `corpus_helpers.metrics` against two external ground truths:
URIEL typological feature distances and a manually specified phylogenetic tree.

**Languages** — chosen to give clear within-family and cross-family signal:
- **West Germanic**: `en` English, `de` German, `nl` Dutch
- **North Germanic**: `da` Danish, `sv` Swedish
- **West Slavic** (Latin script): `cs` Czech, `sk` Slovak, `pl` Polish
- **South Slavic** (Latin script): `hr` Croatian

**Dependency**: `pip install lang2vec` for the URIEL evaluation.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from functools import partial
from corpus_helpers.read import (
    lower, 
    split_line, 
    delete_newline, 
    split_word, 
    delete_blank, 
    as_bytes, 
    chain_preprocessors
)

from corpus_helpers.metrics import (
    ngram_overlap,
    ngram_divergence,
    normalized_compression_distance,
    bpe_overlap, 
)

/Users/rastislavhronsky/Research/corpus-helpers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download Bible XML files

In [2]:
LANGUAGES = {
    'en': 'English',
    'de': 'German',
    'nl': 'Dutch',
    'da': 'Danish',
    'sv': 'Swedish',
    'cs': 'Czech',
    'sk': 'Slovak',
    'pl': 'Polish',
    'hr': 'Croatian',
}
LANG_CODES = list(LANGUAGES.keys())
N = len(LANG_CODES)

RAW_BASE = 'https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles'
CACHE_DIR = Path('_bible_cache')
CACHE_DIR.mkdir(exist_ok=True)

def download(lang_code: str, lang_name: str) -> Path:
    dest = CACHE_DIR / f'{lang_code}.xml'
    if not dest.exists():
        url = f'{RAW_BASE}/{lang_name}.xml'
        print(f'Downloading {lang_name}...', end=' ', flush=True)
        urllib.request.urlretrieve(url, dest)
        print('done')
    else:
        print(f'{lang_name}: cached')
    return dest

paths = {code: download(code, name) for code, name in LANGUAGES.items()}

English: cached
German: cached
Dutch: cached
Danish: cached
Swedish: cached
Czech: cached
Slovak: cached
Polish: cached
Croatian: cached


## 2. Parse XML → list of verse bytes

In [3]:

PRETOKEN_PIPELINE = [lower, split_line, delete_newline, split_word, delete_blank, as_bytes]

def parse_bible(path: Path) -> list[bytes]:
    tree = ET.parse(path)
        # (seg.text or '').strip().encode('utf-8')
    segments = [(seg.text or '').strip() for seg in tree.iter('seg')
                if seg.text and seg.text.strip()]
    return chain_preprocessors(segments, PRETOKEN_PIPELINE)

paths = {code: download(code, name) for code, name in LANGUAGES.items()}
corpora: dict[str, list[bytes]] = {}
for code, path in paths.items():
    corpora[code] = parse_bible(path)
    print(f'{code}: {len(corpora[code]):,} verses')

English: cached
German: cached
Dutch: cached
Danish: cached
Swedish: cached
Czech: cached
Slovak: cached
Polish: cached
Croatian: cached
en: 918,445 verses
de: 820,005 verses
nl: 839,790 verses
da: 786,197 verses
sv: 860,278 verses
cs: 731,091 verses
sk: 746,897 verses
pl: 741,355 verses
hr: 697,515 verses


## 3. Compute pairwise distance matrices

- **NCD** – Normalized Compression Distance (zlib, symmetric). Lower = more similar.
- **JSD** – Jensen-Shannon Divergence on word unigrams. Lower = more similar.
- **Jaccard distance** – 1 − Jaccard overlap of word-unigram vocabularies. Lower = more similar.

In [4]:
import bz2, lzma, gzip

In [ ]:
METRICS = [
    dict(label='ncd_bz2', fn=partial(normalized_compression_distance, compressor=bz2.compress, symmetric=True)),
    dict(label='ncd_lzma', fn=partial(normalized_compression_distance, compressor=lzma.compress, symmetric=True)),
    dict(label='jsd_word_n=1', fn=partial(ngram_divergence, method='jsd', n=1, unit='word')),
    dict(label='jsd_char_n=3', fn=partial(ngram_divergence, method='jsd', n=3, unit='char')),
    dict(label='jacc_word', fn=lambda a, b: 1.0 - ngram_overlap(a, b, n=1, unit='word')),
    dict(label='bpe_overlap', fn=partial(bpe_overlap, min_freq_fract=1e-6, show_progress=False)),
]

LANG_CODES = list(LANGUAGES.keys())
N = len(LANG_CODES)

METRIC_MATS = {m['label']: np.zeros((N, N)) for m in METRICS}
pairs = list(combinations(range(N), 2))
print(f'Computing {len(pairs)} pairs × {len(METRICS)} metrics...')

for i, j in pairs:
    ca, cb = corpora[LANG_CODES[i]], corpora[LANG_CODES[j]]
    print(f"  {LANG_CODES[i]}-{LANG_CODES[j]}", end='  ', flush=True)
    for m in METRICS:
        val = m['fn'](ca, cb)
        METRIC_MATS[m['label']][i, j] = val
        METRIC_MATS[m['label']][j, i] = val
        print(f"{m['label']}={val:.4f}", end='  ', flush=True)
    print()

print('Done.')

Computing 36 pairs × 6 metrics...
  en-de  ncd_bz2=1.0197  ncd_lzma=0.9986  jsd_word_n=1=0.5094  jsd_char_n=3=0.1656  jacc_word=0.9547  bpe_overlap=0.2960  
  en-nl  ncd_bz2=1.0188  ncd_lzma=0.9988  jsd_word_n=1=0.5179  jsd_char_n=3=0.1602  jacc_word=0.9534  bpe_overlap=0.3251  
  en-da  ncd_bz2=1.0143  ncd_lzma=0.9991  jsd_word_n=1=0.4975  jsd_char_n=3=0.1752  jacc_word=0.9663  bpe_overlap=0.2908  
  en-sv  ncd_bz2=1.0188  

In [ ]:
import pickle
with open('notebooks/_bible_cache/mats.pkl', 'wb') as f: 
    pickle.dump(METRIC_MATS, f)

## 4. URIEL evaluation

[URIEL](http://www.cs.cmu.edu/~dmortens/uriel.html) encodes typological properties of languages as feature vectors.
`lang2vec` provides convenient access. We fetch two feature sets and compute cosine distances:
- **`fam`** — one-hot language family membership (should closely match the gold tree)
- **`syntax_knn + phonology_knn + inventory_knn`** — typological features interpolated via k-NN

We then measure how well each corpus metric ranks pairs by Spearman ρ against the URIEL distances.

In [9]:
# setuptools ≥ 72 dropped pkg_resources; shim it before lang2vec imports it
import sys, types, importlib.util, os as _os
if 'pkg_resources' not in sys.modules:
    _pr = types.ModuleType('pkg_resources')
    def _resource_filename(pkg, path):
        spec = importlib.util.find_spec(pkg)
        return _os.path.join(_os.path.dirname(spec.origin), path)
    _pr.resource_filename = _resource_filename
    sys.modules['pkg_resources'] = _pr

import lang2vec.lang2vec as l2v
from sklearn.metrics.pairwise import cosine_distances

# ISO 639-3 codes expected by lang2vec
ISO3 = {
    'en': 'eng', 'de': 'deu', 'nl': 'nld', 'da': 'dan', 'sv': 'swe',
    'cs': 'ces', 'sk': 'slk', 'pl': 'pol', 'hr': 'hrv',
}
iso3_codes = [ISO3[c] for c in LANG_CODES]

URIEL_SETS = {
    'family (fam)':  'fam',                                    # one-hot language family membership
    'typological':   'syntax_knn+phonology_knn+inventory_knn', # interpolated typological features
}

uriel_mats: dict[str, np.ndarray] = {}
for name, fset in URIEL_SETS.items():
    feats = l2v.get_features(iso3_codes, fset)
    vecs = np.array([feats[c] for c in iso3_codes], dtype=float)
    # drop dimensions where any language has NaN
    valid = ~np.any(np.isnan(vecs), axis=0)
    vecs_clean = vecs[:, valid]
    print(f'{name}: {vecs.shape[1]} dims → {valid.sum()} after NaN filter')
    uriel_mats[name] = cosine_distances(vecs_clean)

# Quick sanity check: lowest family distances should be within-family
print()
print('Lowest family distances (should all be within-family):')
fam = uriel_mats['family (fam)']
idx = np.triu_indices(N, k=1)
order = np.argsort(fam[idx])
for rank, o in enumerate(order[:5]):
    i, j = idx[0][o], idx[1][o]
    print(f'  {rank+1}. {LANG_CODES[i]}-{LANG_CODES[j]}  d={fam[i,j]:.4f}')

family (fam): 3718 dims → 3718 after NaN filter
typological: 289 dims → 289 after NaN filter

Lowest family distances (should all be within-family):
  1. cs-sk  d=0.0871
  2. da-sv  d=0.1667
  3. sk-pl  d=0.2000
  4. cs-pl  d=0.2697
  5. de-nl  d=0.3196


In [ ]:
def upper_tri(mat: np.ndarray) -> np.ndarray:
    i, j = np.triu_indices(len(mat), k=1)
    return mat[i, j]

# Spearman ρ between each corpus metric and each URIEL distance
rows = []
for metric_name, metric_mat in METRIC_MATS.items():
    for uriel_name, uriel_mat in uriel_mats.items():
        r, p = spearmanr(upper_tri(metric_mat), upper_tri(uriel_mat))
        rows.append({'metric': metric_name, 'URIEL': uriel_name, 'ρ': r, 'p': p})

df_corr = pd.DataFrame(rows).pivot(index='metric', columns='URIEL', values='ρ')
print('Spearman ρ (corpus metric vs URIEL distance):')
display(df_corr.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1).format('{:.3f}'))

# p-values
df_p = pd.DataFrame(rows).pivot(index='metric', columns='URIEL', values='p')
print('\np-values:')
display(df_p.style.format('{:.4f}'))

In [ ]:
# Scatter: each metric vs each URIEL distance, one subplot per combination
fig, axes = plt.subplots(
    len(uriel_mats), len(METRIC_MATS),
    figsize=(5 * len(METRIC_MATS), 4 * len(uriel_mats)),
    squeeze=False,
)

pair_labels = [f'{LANG_CODES[i]}-{LANG_CODES[j]}'
               for i, j in zip(*np.triu_indices(N, k=1))]
family_colors = [
    'tab:blue' if LANG_CODES[i] in ('en','de','nl','da','sv')
               and LANG_CODES[j] in ('en','de','nl','da','sv')
    else 'tab:red' if LANG_CODES[i] in ('cs','sk','pl','hr')
                   and LANG_CODES[j] in ('cs','sk','pl','hr')
    else 'tab:grey'
    for i, j in zip(*np.triu_indices(N, k=1))
]

for row, (uriel_name, uriel_mat) in enumerate(uriel_mats.items()):
    u = upper_tri(uriel_mat)
    for col, (metric_name, metric_mat) in enumerate(METRIC_MATS.items()):
        ax = axes[row, col]
        m = upper_tri(metric_mat)
        r, _ = spearmanr(m, u)
        ax.scatter(m, u, c=family_colors, s=40, alpha=0.8, zorder=2)
        for lbl, mx, uy in zip(pair_labels, m, u):
            ax.annotate(lbl, (mx, uy), fontsize=6, ha='left', va='bottom',
                        xytext=(3, 2), textcoords='offset points')
        ax.set_xlabel(metric_name)
        ax.set_ylabel(f'URIEL {uriel_name}')
        ax.set_title(f'ρ = {r:.3f}')
        ax.grid(True, alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue',  label='within Germanic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:red',   label='within Slavic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:grey',  label='cross-family'),
]
fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0))
fig.suptitle('Corpus metric vs URIEL distance', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('bible_uriel_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Dendrogram vs. gold phylogenetic tree

We encode the standard IE sub-tree for our 9 languages as a cophenetic distance matrix,
then compare each metric's average-linkage dendrogram to it.

**Gold tree** (simplified, branch depths are ordinal):
```
Indo-European (depth 4)
├── Germanic (depth 2)
│   ├── West Germanic (depth 1): en, de, nl
│   └── North Germanic (depth 1): da, sv
└── Slavic (depth 2)
    ├── West Slavic (depth 1): cs, sk, pl
    └── South Slavic (depth 1): hr  (single-member; merges at depth 2)
```
Cophenetic distances = depth of the lowest common ancestor.

In [ ]:
# LANG_CODES order: en de nl da sv cs sk pl hr
# Group membership
W_GERM  = {'en', 'de', 'nl'}   # depth-1 siblings
N_GERM  = {'da', 'sv'}         # depth-1 siblings
GERMANIC = W_GERM | N_GERM     # depth-2 siblings
W_SLAV  = {'cs', 'sk', 'pl'}   # depth-1 siblings
S_SLAV  = {'hr'}               # depth-1 (solo); merges at depth 2
SLAVIC   = W_SLAV | S_SLAV     # depth-2 siblings

def gold_distance(a: str, b: str) -> int:
    if {a, b} <= W_GERM or {a, b} <= N_GERM or {a, b} <= W_SLAV:
        return 1   # same sub-branch
    if {a, b} <= GERMANIC or {a, b} <= SLAVIC:
        return 2   # same branch, different sub-branch
    return 4       # cross-family (IE root)

gold_mat = np.array(
    [[gold_distance(a, b) for b in LANG_CODES] for a in LANG_CODES],
    dtype=float,
)

print('Gold cophenetic distance matrix:')
df_gold = pd.DataFrame(gold_mat, index=LANG_CODES, columns=LANG_CODES)
display(df_gold.style.background_gradient(cmap='YlOrRd').format('{:.0f}'))

In [ ]:
def make_linkage(mat: np.ndarray) -> np.ndarray:
    return linkage(squareform(mat), method='average')

Z_gold   = make_linkage(gold_mat)
Z_metric = {name: make_linkage(mat) for name, mat in METRIC_MATS.items()}

# Cophenetic distances from each tree
coph_gold = cophenet(Z_gold)   # condensed-form cophenetic distances
coph_metrics = {name: cophenet(Z) for name, Z in Z_metric.items()}

# Spearman correlation between metric tree and gold tree (via cophenetic distances)
print('Cophenetic correlation with gold tree (Spearman ρ on cophenetic distances):')
for name, coph in coph_metrics.items():
    r, p = spearmanr(coph, coph_gold)
    print(f'  {name:10s}  ρ = {r:.4f}  p = {p:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Gold tree panel
ax = axes[0]
dendrogram(
    Z_gold, labels=LANG_CODES, ax=ax,
    leaf_rotation=45, color_threshold=3,
    link_color_func=lambda k: 'tab:blue' if k < 5 else ('tab:red' if k < 9 else 'tab:grey'),
)
ax.set_title('Gold tree (IE phylogeny)', fontweight='bold')
ax.set_ylabel('depth')

# One panel per metric
for ax, (name, Z) in zip(axes[1:], Z_metric.items()):
    r, _ = spearmanr(coph_metrics[name], coph_gold)
    dendrogram(
        Z, labels=LANG_CODES, ax=ax,
        leaf_rotation=45,
        color_threshold=0.7 * max(Z[:, 2]),
    )
    ax.set_title(f'{name}  (ρ={r:.3f} vs gold)')
    ax.set_ylabel('distance')

plt.suptitle('Metric dendrograms vs. gold phylogenetic tree (Bible corpus)', fontsize=13)
plt.tight_layout()
plt.savefig('bible_dendrograms_vs_gold.png', dpi=150, bbox_inches='tight')
plt.show()